# 01 — Data Exploration
Explore raw Statcast data: distributions, coordinate systems, season counts.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

In [ ]:
# Load one season for quick exploration
df = pd.read_csv('../data/raw/statcast_2023.csv', low_memory=False)
print(df.shape)
df.head(3)

In [ ]:
# Landing coordinate distributions
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(df['hc_x'].dropna(), bins=80, color='steelblue', edgecolor='none')
axes[0].set_xlabel('hc_x (Statcast pixel)')
axes[0].set_title('X distribution')
axes[1].hist(df['hc_y'].dropna(), bins=80, color='tomato', edgecolor='none')
axes[1].set_xlabel('hc_y (Statcast pixel, 0=top)')
axes[1].set_title('Y distribution (raw, y=0 is top)')
fig.tight_layout()

In [ ]:
# 2D scatter of all batted ball landing locations (raw pixel space)
sample = df[['hc_x', 'hc_y']].dropna().sample(5000, random_state=42)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(sample['hc_x'], -sample['hc_y'], s=1, alpha=0.3, c='steelblue')
ax.set_title('Raw Statcast coords (y-flipped so outfield is up)')
ax.set_aspect('equal')
ax.set_xlabel('hc_x'); ax.set_ylabel('-hc_y')

In [ ]:
# Plate appearances per batter
pa_per_batter = df.groupby('batter').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(pa_per_batter, bins=50, color='darkgreen', edgecolor='none')
ax.axvline(100, color='red', linestyle='--', label='Min PA (full-season)')
ax.axvline(30, color='orange', linestyle='--', label='Min PA (situational)')
ax.set_xlabel('Plate appearances'); ax.set_ylabel('# batters')
ax.set_title('PA distribution per batter (2023)')
ax.legend()
print(f'Batters with >=100 PA: {(pa_per_batter>=100).sum()}')
print(f'Batters with >=30 PA : {(pa_per_batter>=30).sum()}')

In [ ]:
# Pitch type distribution
from src.data.preprocess import PITCH_TYPE_MAP
df['pitch_group'] = df['pitch_type'].map(PITCH_TYPE_MAP)
df['pitch_group'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'])
plt.title('Pitch type groups'); plt.ylabel('# events'); plt.tight_layout()

In [ ]:
# Count state distribution
from src.data.preprocess import count_state
df['count_state'] = df.apply(lambda r: count_state(int(r['balls']), int(r['strikes'])), axis=1)
df['count_state'].value_counts().plot(kind='bar', color='slateblue')
plt.title('Count state distribution'); plt.tight_layout()

In [ ]:
# Preview one full-season spray chart image
import numpy as np
from src.data.preprocess import normalize_coords, coords_to_image
from src.analysis.visualize import plot_spray_chart

top_batter = pa_per_batter.index[0]
grp = df[df['batter'] == top_batter].dropna(subset=['hc_x','hc_y'])
x, y = normalize_coords(grp['hc_x'].values, grp['hc_y'].values)
img = coords_to_image(x, y)

fig, ax = plt.subplots(figsize=(4, 4))
plot_spray_chart(img, ax=ax, title=f'Batter {top_batter} — 2023 spray chart')
plt.tight_layout()